# 🕐 Notebook 7: Context Features — Temporal & Demographics

**⚠️ Chạy notebook 00 hoặc 01 trước để có data!**

In [ ]:
import subprocess, sys, os

try:
    import surprise
    import numpy as np
    import pandas as pd
    assert int(np.__version__.split('.')[0]) < 2, 'need numpy<2'
    assert int(pd.__version__.split('.')[0]) < 3, 'need pandas<3'
    print(f'✅ OK (numpy={np.__version__}, pandas={pd.__version__}, surprise={surprise.__version__})')
except Exception as e:
    print(f'📦 Installing... ({e})')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install',
        'numpy<2', 'pandas<3', 'scikit-surprise', 'scikit-learn',
        'matplotlib', 'seaborn', 'tqdm', '-q'])
    print('✅ Install xong! Runtime đang restart...')
    os.kill(os.getpid(), 9)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

sns.set(style='whitegrid')
os.makedirs('results/charts', exist_ok=True)

ratings = pd.read_csv('data/processed/ratings_clean.csv')
movies  = pd.read_csv('data/processed/movies_clean.csv')
users   = pd.read_csv('data/processed/users_clean.csv')

# Đảm bảo columns tồn tại
age_map = {1:'Under 18',18:'18-24',25:'25-34',35:'35-44',45:'45-49',50:'50-55',56:'56+'}
if 'age_group' not in users.columns:
    users['age_group'] = users['age'].map(age_map)

print(f'✅ Loaded: {len(ratings):,} ratings, {len(users):,} users')

## A. Temporal Analysis

In [ ]:
ratings['datetime'] = pd.to_datetime(ratings['timestamp'], unit='s')
ratings['year']      = ratings['datetime'].dt.year
ratings['dayofweek'] = ratings['datetime'].dt.dayofweek
ratings['hour']      = ratings['datetime'].dt.hour

# Rating theo ngày trong tuần
day_names = {0:'Mon',1:'Tue',2:'Wed',3:'Thu',4:'Fri',5:'Sat',6:'Sun'}
day_stats = ratings.groupby('dayofweek')['rating'].mean()

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar([day_names[i] for i in day_stats.index], day_stats.values, color=['steelblue']*5 + ['coral','coral'], edgecolor='black')
ax.set_title('Rating trung bình theo ngày trong tuần')
ax.set_ylabel('Avg Rating')
ax.set_ylim(3.5, 4.0)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('results/charts/07_temporal_by_dayofweek.png', dpi=150, bbox_inches='tight')
plt.show()

## B. Demographics Analysis

In [ ]:
merged = ratings.merge(users[['userId','gender','age_group']], on='userId')

# Gender
gender_stats = merged.groupby('gender')['rating'].mean()
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(gender_stats.index, gender_stats.values, color=['coral', 'steelblue'], edgecolor='black')
ax.set_title('Rating trung bình theo giới tính')
ax.set_ylabel('Avg Rating')
ax.set_ylim(3.5, 4.0)
for i, v in enumerate(gender_stats.values):
    ax.text(i, v + 0.005, f'{v:.3f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('results/charts/07_demographics_gender.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Genre preferences by gender
merged = merged.merge(movies[['movieId','genres']], on='movieId')
all_genres = ['Action','Adventure','Animation',"Children's",'Comedy','Crime','Documentary','Drama','Fantasy','Film-Noir','Horror','Musical','Mystery','Romance','Sci-Fi','Thriller','War','Western']

results = []
for g in all_genres:
    merged[f'is_{g}'] = merged['genres'].str.contains(g, na=False).astype(int)
    male = merged[(merged['gender']=='M') & (merged[f'is_{g}']==1)]['rating'].mean()
    female = merged[(merged['gender']=='F') & (merged[f'is_{g}']==1)]['rating'].mean()
    results.append({'genre': g, 'male': male, 'female': female, 'diff': female - male})

genre_gender = pd.DataFrame(results).sort_values('diff', ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(genre_gender))
ax.bar(x - 0.175, genre_gender['male'], 0.35, label='Male', color='steelblue')
ax.bar(x + 0.175, genre_gender['female'], 0.35, label='Female', color='coral')
ax.set_xticks(x)
ax.set_xticklabels(genre_gender['genre'], rotation=30, ha='right')
ax.set_title('Genre Preferences: Male vs Female')
ax.set_ylim(3.0, 4.2)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('results/charts/07_genre_by_gender.png', dpi=150, bbox_inches='tight')
plt.show()

## Tổng kết

- Cuối tuần rating cao hơn ngày thường
- Male: Action/Sci-Fi, Female: Romance/Drama
- Context features giúp cải thiện recommendation